Se conservarán las columnas con los precios originales del EUR/USD, ya que serán útiles para revisar los datos y comprender posteriormente los resultados obtenidos por el modelo.

Sin embargo, no se utilizará directamente el precio bruto del EUR/USD como variable de entrada para realizar las predicciones. Esto se debe a que el valor del par cambia con el paso del tiempo y un mismo nivel de precio puede tener un significado diferente dependiendo del periodo en el que se encuentre el mercado. Por ejemplo, un precio de 1,10 puede representar una situación distinta según el contexto histórico en el que se haya producido.

En lugar de utilizar únicamente el nivel del precio, se crearán variables que permitan describir de mejor manera el comportamiento reciente del mercado. Entre estas variables se incluirán los retornos, que indican cuánto ha subido o bajado el precio; las tendencias, que permiten identificar la dirección general del movimiento; la volatilidad, que muestra cuánto varía el precio; y otros indicadores relativos calculados a partir de los datos históricos.

De esta manera, se buscará que el modelo no aprenda simplemente que un precio específico significa que el EUR/USD va a subir o bajar, sino que analice cómo se está comportando el mercado en ese momento.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Localizo la carpeta principal
ruta_actual = Path.cwd().resolve()

if ruta_actual.name == "notebooks":
    ruta_proyecto = ruta_actual.parent
else:
    ruta_proyecto = ruta_actual

ruta_processed = ruta_proyecto / "data" / "processed"

# Cargo el dataset limpio
archivo_limpio = ruta_processed / "eurusd_yahoo_limpio.csv"

datos = pd.read_csv(
    archivo_limpio,
    parse_dates=["Date"],
    index_col="Date"
)

datos["Objetivo"] = datos["Objetivo"].astype("Int64")

print("Filas cargadas:", len(datos))
print("Primera fecha:", datos.index.min().date())
print("Última fecha:", datos.index.max().date())

Filas cargadas: 5845
Primera fecha: 2004-01-01
Última fecha: 2026-07-14


In [2]:
# Trabajo sobre una copia
datos_modelo = datos.copy()

# Retornos de jornadas anteriores
# Uso rezagos de 1, 2, 3 y 5 jornadas para representar movimientos recientes y de una semana, asi evito variables muy parecidas
for rezago in [1, 2, 3, 5]:
    datos_modelo[f"Retorno_lag_{rezago}"] = (
        datos_modelo["Retorno_diario"].shift(rezago) # Guardo los retornos de jornadas anteriores para que el modelo use información pasada
    )

# Información de la vela diaria
datos_modelo["Rango_diario"] = ( # me ssirve para saber cuanto se movio el precio, pero no me indica si cerro en alza o baja respecto a la apertura
    (datos_modelo["High"] - datos_modelo["Low"])
    / datos_modelo["Open"] # Al dividirlo entre Open, sabemos qué tan grande fue ese movimiento en relación con el precio de ese día
                           # aqui saco un porcentaje de cambio en relacion al precio de apertura
)

datos_modelo["Cuerpo_vela"] = ( # solo compruebo jornadas alcitas o bajistas
    (datos_modelo["Close"] - datos_modelo["Open"])
    / datos_modelo["Open"]
)

rango = datos_modelo["High"] - datos_modelo["Low"]

datos_modelo["Posicion_cierre"] = np.where(
    rango != 0, # aqui me estoy asegurando que haya un rango
    (datos_modelo["Close"] - datos_modelo["Low"]) / rango,
    0.5 # aqui me estaba dando problemas porque existian divisiones para 0 y me arrojaba una incongruencia, por eso mejor
        # cuando no existe rango diario, se asigna una posición neutral de 0.5 para evitar eliminar la jornada
        # como calculaba en que parte del rango diario quedó el cierre, 0.5 no favorece a ningun extremo
)

# Distancia del cierre respecto a medias móviles (SMA porque son medias moviles simples)
for ventana in [5, 10, 20]:
    media_movil = datos_modelo["Close"].rolling( # al hacer un promedio con 5,10 y 20 jornadas puedo identificar tendencias mas claras y
                                                 # saber diferenciar una tendencia de un cambio reciente
        window=ventana
    ).mean()

    datos_modelo[f"Distancia_MA{ventana}"] = ( # no guardo directamente la media porque ese valor tiene el mismo problema que el 
                                               # precio original (depende del nivel histórico del EUR/USD)
        datos_modelo["Close"] / media_movil - 1 # le resto 1 porque solo me interesa la distancia y no la proporcion respecto del promedio
    )# analizando las distancias entre las medias puedo observar una tendencia o cambios bruscos

# Volatilidad reciente
for ventana in [5, 20]:
    datos_modelo[f"Volatilidad_{ventana}"] = (
        datos_modelo["Retorno_diario"]
        .rolling(window=ventana)
        .std() # saco la desviacion estandar para ver que tan dispersos están los retornos
    )
    
# RSI de 14 jornadas
cambio = datos_modelo["Close"].diff() # diff resta el cierre anterior al cierre actual
                                      # si hiciera el promedio sobre los precios de cierre solo obtendria el precio promedio
                                      # y a mi me interesa el promedio de cuanto subió o bajó
                                      # RSI compara el promedio de la diferencia de los precios de cierre, no el promedio de los cierres

ganancias = cambio.clip(lower=0)
perdidas = -cambio.clip(upper=0) # el signo es para volver positivos los negativos porque solo me interesa el valor absoluto

media_ganancias = ganancias.rolling(window=14).mean() #calculo el promedio de las ultimas 14 jornadas porque ese es el estandar de RSI
media_perdidas = perdidas.rolling(window=14).mean()

rs = media_ganancias / media_perdidas #rs significa fuerza relativa

datos_modelo["RSI_14"] = 100 - (100 / (1 + rs)) # es la fórmula estándar del RSI para transformar la relación 
                                                # entre las ganancias y las pérdidas en una escala de 0 a 100
# RSI = 75: las subidas estan dominando   
# RSI = 50: equilibrio entre subidas y bajadas
# RSI = 25: las bajadas estn dominando                                      
                                                


# MACD - es un indicador que compara medias moviles para medir el impulso y la fuerza de la tendencia del precio de un activo
# 
# EMA(medias moviles exponenciales)
# 
# La media movil exponencial permite representar la tendencia del precio dando mayor importancia a los cierres recientes. La 
# media móvil simple asigna el mismo peso a todas las jornadas, por lo que una ema responde más rápidamente ante cambios nuevos del precio
# el valor de span no representa la cantidad de jornadas que se están calculando, sino la sensiblidad que tiene la ema cada que aparece
# un precio de cierre nuevo, como cada ema nueva se calcula utilizando la ema anterior, termina utilizando todas las jornadas del dataset
# de manera implicita (el cierre más antiguo termina teniendo un peso extremadamente pequeño)

ema_12 = datos_modelo["Close"].ewm(
    span=12,
    adjust=False
).mean()

ema_26 = datos_modelo["Close"].ewm(
    span=26,
    adjust=False
).mean()

macd = ema_12 - ema_26 # al resta la ema rapida de la lenta miro qué tan separada está la tendencia reciente de la tendencia más lenta
# cuanto mas lejos de cero está el valor de la diferencia de las emas, mayor es la intensidad del impulso reciente

senal_macd = macd.ewm(
    span=9,
    adjust=False
).mean()

datos_modelo["MACD_hist"] = macd - senal_macd

# El MACD indica la dirección y la intensidad """"ACTUAL (en ese momento)"""" del impulso, 
# mientras que el histograma muestra si ese impulso se está fortaleciendo o debilitando, lo que puede anticipar un posible cambio de dirección

# MACD positivo = impulso alcista
# Está por encima de la señal (MACD_hist positivo) = el impulso alcista se está fortaleciendo

# MACD positivo = el impulso siguen siendo alcista
# Pero está debajo de la señal: ese impulso alcista se está debilitando

# ------------ La intensidad del impulso reciente indica con qué fuerza el precio se está moviendo ESE MOMENTO hacia arriba o hacia abajo, 
# no con qué fuerza va toda la tendencia general 




In [3]:
variables_predictoras = [
    "Retorno_diario",
    "Retorno_lag_1",
    "Retorno_lag_2",
    "Retorno_lag_3",
    "Retorno_lag_5",
    "Rango_diario",
    "Cuerpo_vela",
    "Posicion_cierre",
    "Distancia_MA5",
    "Distancia_MA10",
    "Distancia_MA20",
    "Volatilidad_5",
    "Volatilidad_20",
    "RSI_14",
    "MACD_hist"
]

# Se seleccionaron 15 variables porque cada una aporta información diferente sobre el comportamiento del EUR/USD, como los movimientos 
# anteriores, la forma de la vela, la tendencia, la volatilidad y el impulso. Más adelante se evaluará su utilidad para comprobar si todas 
# aportan información al modelo o si alguna resulta innecesaria

print("Número de variables predictoras:", len(variables_predictoras))
print(variables_predictoras)

Número de variables predictoras: 15
['Retorno_diario', 'Retorno_lag_1', 'Retorno_lag_2', 'Retorno_lag_3', 'Retorno_lag_5', 'Rango_diario', 'Cuerpo_vela', 'Posicion_cierre', 'Distancia_MA5', 'Distancia_MA10', 'Distancia_MA20', 'Volatilidad_5', 'Volatilidad_20', 'RSI_14', 'MACD_hist']


In [4]:
# Reviso qué variables generan filas incompletas
faltantes_variables = (
    datos_modelo[variables_predictoras]
    .isna()
    .sum()
)

print(faltantes_variables[faltantes_variables > 0])

Retorno_diario     1
Retorno_lag_1      2
Retorno_lag_2      3
Retorno_lag_3      4
Retorno_lag_5      6
Distancia_MA5      4
Distancia_MA10     9
Distancia_MA20    19
Volatilidad_5      5
Volatilidad_20    20
RSI_14            14
dtype: int64


In [5]:
filas_antes = len(datos_modelo)


# Elimino únicamente filas sin variables predictoras completas
datos_modelo = datos_modelo.dropna(
    subset=variables_predictoras
).copy()


filas_despues = len(datos_modelo)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Filas eliminadas:", filas_antes - filas_despues)

# Las primeras jornadas no tendrán medias móviles ni volatilidad de 20 días, porque todavía no existen suficientes jornadas anteriores
# Se eliminan esas primeras jornadas y quizas algunas jornadas donde High y Low eran iguales, 
# por lo que Posicion_cierre quedó como NaN

Filas antes: 5845
Filas después: 5825
Filas eliminadas: 20


In [6]:
columnas_base = [
    "Open",
    "High",
    "Low",
    "Close"
]

columnas_finales = (
    columnas_base
    + variables_predictoras
    + ["Objetivo"]
)

datos_modelo = datos_modelo[columnas_finales]

# Selecciono únicamente las columnas necesarias para el análisis y el entrenamiento del modelo (elimino las columnas auxiliares porque
# ya tengo el valor que necesitaba)

In [7]:
if datos_modelo[variables_predictoras].isna().any().any():
    raise ValueError(
        "Todavía existen valores faltantes en las variables predictoras."
    )

print("Dimensiones:", datos_modelo.shape)
print(
    "Filas disponibles para entrenar:",
    datos_modelo["Objetivo"].notna().sum()
)
print(
    "Filas reservadas para predicción futura:",
    datos_modelo["Objetivo"].isna().sum()
)

display(datos_modelo.tail())

#Compruebo que las variables predictoras no tengan valores faltantes y se muestran las dimensiones finales del dataset. 
# También indico cuántas filas pueden utilizarse para entrenar el modelo y cuántas quedan reservadas para realizar una predicción futura

Dimensiones: (5825, 20)
Filas disponibles para entrenar: 5824
Filas reservadas para predicción futura: 1


,Open,High,Low,Close,Retorno_diario,Retorno_lag_1,Retorno_lag_2,Retorno_lag_3,Retorno_lag_5,Rango_diario,Cuerpo_vela,Posicion_cierre,Distancia_MA5,Distancia_MA10,Distancia_MA20,Volatilidad_5,Volatilidad_20,RSI_14,MACD_hist,Objetivo
Date,,,,,,,,,,,,,,,,,,,,
2026-07-08,1.140095,1.143249,1.139186,1.140381,-0.003318,0.000355,0.001315,0.003929,-0.000765,0.003564,0.000251,0.294130,-0.001138,0.000151,-0.004795,0.003072,0.003207,36.201358,0.000624,1
2026-07-09,1.142191,1.145082,1.142126,1.142204,0.001599,-0.003318,0.000355,0.001315,-0.003095,0.002588,0.000011,0.026458,-0.000312,0.001155,-0.002709,0.002639,0.003242,44.702069,0.000749,1
2026-07-10,1.143380,1.146263,1.141839,1.143341,0.000995,0.001599,-0.003318,0.000355,0.003929,0.003869,-0.000034,0.339360,0.000495,0.001521,-0.001096,0.002015,0.003126,45.827333,0.000910,0
2026-07-13,1.140368,1.144571,1.138680,1.140446,-0.002532,0.000995,0.001599,-0.003318,0.001315,0.005166,0.000068,0.299802,-0.001457,-0.001179,-0.002759,0.002203,0.003073,46.714612,0.000825,0
2026-07-14,1.138369,1.146132,1.137877,1.138433,-0.001765,-0.002532,0.000995,0.001599,0.000355,0.007252,0.000057,0.067466,-0.002215,-0.002613,-0.003605,0.002182,0.003079,50.754357,0.000644,<NA>


In [ ]:
archivo_variables = (
    ruta_processed / "eurusd_yahoo_variables.csv"
)

datos_modelo.to_csv(
    archivo_variables,
    index=True,
    encoding="utf-8"
)

print("Dataset guardado en:")
print(archivo_variables)

Dataset guardado en:
C:\TFM_EURUSD\data\processed\eurusd_variables_modelo.csv
